# Phase 0 — 로직체인 탐색
생성된 로직체인을 카테고리별로 확인하고 ChromaDB 검색 테스트

In [2]:
import json
import pandas as pd
from pathlib import Path

CHAINS_DIR = Path('data/logic_chains')

# 전체 로드
with open(CHAINS_DIR / 'all_chains.json', encoding='utf-8') as f:
    all_chains = json.load(f)

print(f'총 로직체인 수: {len(all_chains)}')
print(f'카테고리 수: {len(set(c["category"] for c in all_chains))}')

총 로직체인 수: 475
카테고리 수: 19


In [3]:
# 카테고리별 개수
df = pd.DataFrame(all_chains)
print(df.groupby('category').size().sort_values(ascending=False).to_string())

category
AI_기술혁신        25
ESG_기후변화       25
MA_기업이벤트       25
공급망_물류         25
금_안전자산         25
금리_통화정책        25
금융위기_신용경색      25
기술_규제          25
노동시장_고용        25
무역_관세          25
미국_선거_정치       25
신흥국_위기         25
실적_어닝시즌        25
암호화폐_디지털자산     25
원자재_에너지        25
인플레이션_디플레이션    25
지정학_전쟁         25
테러_자연재해        25
팬데믹_전염병        25


In [10]:
# 카테고리 선택해서 보기
CATEGORY = '지정학_전쟁'   # ← 여기 바꿔보기

chains = [c for c in all_chains if c['category'] == CATEGORY]
print(f'[{CATEGORY}] {len(chains)}개\n')

for i, c in enumerate(chains[:]):
    print(f"{'='*60}")
    print(f"[{i+1}] {c['event']}")
    print(f"    인과경로: {c['causal_path']}")
    print(f"    수혜섹터: {', '.join(c['beneficiary_sectors'])}")
    print(f"    피해섹터: {', '.join(c['victim_sectors'])}")
    print(f"    강도: {c['intensity']} | 반응속도: {c['reaction_speed']}")
    print(f"    선행징후: {c['pre_signals'][0] if c['pre_signals'] else '-'}")

[지정학_전쟁] 25개

[1] 이란의 호르무즈 해협 전면 봉쇄
    인과경로: 해협 봉쇄 선언 → 글로벌 원유 물동량 20% 차단 → 국제 유가 폭등 및 인플레이션 우려 점화 → 에너지 섹터 급등 및 운송/소비재 타격
    수혜섹터: Oil & Gas Exploration, Oilfield Services, Marine Shipping
    피해섹터: Airlines, Cruise Lines, Consumer Discretionary
    강도: high | 반응속도: 즉각반응
    선행징후: 이란 혁명수비대의 해상 기동 훈련 빈도 급증
[2] 이스라엘의 이란 핵시설 선제 타격
    인과경로: 핵시설 공습 → 이란의 즉각적인 보복 선언 및 중동 전면전 공포 → 글로벌 안전자산 선호 현상 극대화 → 방산주 및 금 관련주 급등
    수혜섹터: Aerospace & Defense, Gold Miners, Cybersecurity
    피해섹터: High Beta Tech, Semiconductors, Regional Banks
    강도: high | 반응속도: 즉각반응
    선행징후: 이스라엘 전시 내각의 강경 발언 및 비상 회의 소집
[3] 후티 반군의 홍해 대형 유조선 격침
    인과경로: 유조선 격침 → 수에즈 운하 통항 전면 중단 및 희망봉 우회 장기화 → 해상 운임 폭등 및 공급망 병목 현상 → 해운주 급등
    수혜섹터: Marine Shipping, Logistics & Freight, Oil & Gas Refining
    피해섹터: Retailers, Auto Manufacturers, Apparel & Luxury Goods
    강도: medium | 반응속도: 1~3일
    선행징후: 홍해 인근에서 드론 및 미사일 요격 횟수 증가
[4] 중동(가자/레바논) 전면 휴전 협정 타결
    인과경로: 휴전 공식 발표 → 지정학적 리스크 프리미엄 소멸 → 유가 하락 및 인플레이션 우려 완화 → 여행/항공주 급등 및 에너지

In [5]:
# 전체 카테고리 목록
categories = sorted(set(c['category'] for c in all_chains))
for cat in categories:
    n = len([c for c in all_chains if c['category'] == cat])
    print(f'  {cat}: {n}개')

  AI_기술혁신: 25개
  ESG_기후변화: 25개
  MA_기업이벤트: 25개
  공급망_물류: 25개
  금_안전자산: 25개
  금리_통화정책: 25개
  금융위기_신용경색: 25개
  기술_규제: 25개
  노동시장_고용: 25개
  무역_관세: 25개
  미국_선거_정치: 25개
  신흥국_위기: 25개
  실적_어닝시즌: 25개
  암호화폐_디지털자산: 25개
  원자재_에너지: 25개
  인플레이션_디플레이션: 25개
  지정학_전쟁: 25개
  테러_자연재해: 25개
  팬데믹_전염병: 25개


In [6]:
# 강도별 분포
print('강도별:')
print(df['intensity'].value_counts().to_string())
print()
print('반응속도별:')
print(df['reaction_speed'].value_counts().to_string())

강도별:
intensity
high      270
medium    195
low        10

반응속도별:
reaction_speed
즉각반응    352
1~3일    108
1주일+     15


## ChromaDB 벡터 검색 테스트
현재 시장 신호를 입력하면 가장 유사한 로직체인을 찾아줌

In [7]:
import sys
sys.path.insert(0, '.')

from src.phase0.embed_chains import search_chains

# ← 여기 현재 시장 상황으로 바꿔서 테스트
QUERY = '미국 국채 10년물 수익률 급등, 달러인덱스 강세, VIX 상승'

results = search_chains(query=QUERY, n_results=5, reaction_speed=None)

print(f'쿼리: "{QUERY}"\n')
for i, r in enumerate(results):
    print(f"[{i+1}] 유사도: {r['similarity']:.3f}")
    print(f"    이벤트: {r['event']}")
    print(f"    인과경로: {r['causal_path']}")
    print(f"    수혜섹터: {', '.join(r['beneficiary_sectors'])}")
    print(f"    반응속도: {r['reaction_speed']}")
    print()

/usr/local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1601.25it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


쿼리: "미국 국채 10년물 수익률 급등, 달러인덱스 강세, VIX 상승"

[1] 유사도: 0.591
    이벤트: VIX 지수 30 돌파 (공포 장세)
    인과경로: 주식시장 변동성 극대화 및 투매 발생 → 기관 투자자들의 리스크 오프(Risk-off) 포지셔닝 → 금, 스위스 프랑 등 전통 안전자산 매수세 폭발
    수혜섹터: Gold Miners, Volatility ETFs
    반응속도: 즉각반응

[2] 유사도: 0.580
    이벤트: 공포지수(VIX) 40 이상 돌파 및 변동성 폭발
    인과경로: 불확실성 극대화로 풋옵션 수요 폭발 → 옵션 마켓 메이커들의 델타 헤징(주식 매도) → 기계적 매도 프로그램(CTA) 가동 → 증시 수직 낙하
    수혜섹터: Volatility ETFs, Market Makers, Treasury ETFs
    반응속도: 즉각반응

[3] 유사도: 0.570
    이벤트: 주요 청산소(Clearinghouse)의 증거금 대폭 인상
    인과경로: 시장 변동성 극대화 → 청산소가 파생상품 및 주식 거래 증거금 기습 인상 → 기관 및 개인의 강제 디레버리징(매도) → 증시 폭락
    수혜섹터: Volatility ETFs, Cash Equivalents
    반응속도: 즉각반응

[4] 유사도: 0.526
    이벤트: 러시아의 우크라이나 전술핵 사용
    인과경로: 전술핵 투하 → 제2차 세계대전 이후 첫 핵무기 사용으로 인한 글로벌 패닉 셀링 → 모든 위험자산 투매 및 초안전자산 쏠림 → VIX 및 금광주 폭등
    수혜섹터: Gold Miners, Volatility ETFs, Aerospace & Defense
    반응속도: 즉각반응

[5] 유사도: 0.517
    이벤트: 대어급 AI/테크 기업의 성공적인 IPO (상장 첫날 폭등)
    인과경로: 신규 상장 테크 기업에 대한 시장의 뜨거운 수요 확인 → IPO 시장 빙하기 종료 기대감 → 투자은행(IB) 및 벤처캐피탈, 

In [9]:
# ChromaDB 통계
import chromadb
client = chromadb.PersistentClient(path='data/chromadb')
col = client.get_collection('logic_chains')
print(f'ChromaDB 저장된 벡터 수: {col.count()}')

ChromaDB 저장된 벡터 수: 475
